# Tutorial 3 - Training the bulkdgd model

This tutorial shows how to train a new bulkdgd model from scratch on a custom set of samples, rather than using the pre-trained model shipped with the package. This is useful when you want a model tailored to a specific set of genes or samples.

As a concrete example, we use 50 real GTEx samples (spanning 10 different tissues), provided as a single CSV file - as most real-world datasets will be - which we split into a training and a test set ourselves.

**Note**: this example uses a small, illustrative training/test set and is meant to demonstrate the training API, not to produce a model of publication quality - training a useful model typically requires many more samples and epochs, and the resulting decoder's parameters file can be several hundred MB in size.

In [1]:
# Import libraries.
import logging as log
import os
import numpy as np
import pandas as pd
import torch
from bulkdgd import ioutil
from bulkdgd.core import model

# Set the logging options so that every message of level INFO or above
# is emitted.
log.basicConfig(level = "INFO")

## Load the configuration for the model

Load the (untrained) model configuration, which describes the architecture of the latent space and the decoder to be trained.

In [2]:
# Load the model's configuration.
config_model = ioutil.load_config_model("model.yaml")

## Create the model

Instantiate a new, untrained bulkdgd model from the configuration, and move it to the GPU if one is available.

In [3]:
# Create the bulkDGD model (latent space and decoder).
dgd_model = model.BulkDGD(**config_model)

# If a CPU with CUDA is available.
if torch.cuda.is_available():

    # Set the GPU as the device.
    device = torch.device("cuda")

# Otherwise
else:

    # Set the CPU as the device.
    device = torch.device("cpu")

# Move the model to the device.
dgd_model.device = device

INFO:bulkdgd.core.model:The latent space was successfully set (type: 'GaussianMixtureModelTGMM').


INFO:bulkdgd.core.decoders:The decoder's hidden layer # 1 was set. Input features: 50. Output features: 500. Activation function: 'ReLU'.


INFO:bulkdgd.core.decoders:The decoder's hidden layer # 2 was set. Input features: 500. Output features: 8000. Activation function: 'ReLU'.


INFO:bulkdgd.core.decoders:The decoder's output module was set. Module 'OutputModuleNBFeatureDispersion' (activation = 'softplus', r_init = 2, output_dim = 14895).


INFO:bulkdgd.core.decoders:The decoder's dropout fraction was set to 0.0.


INFO:bulkdgd.core.model:The decoder was successfully set.


## Load the samples

Load all the samples from a single CSV file - this is how most real-world datasets will be provided - so that we can split it into training and test sets ourselves in the next step.

In [4]:
# Load all the samples into a data frame.
df_samples_raw = \
    ioutil.load_samples(# The CSV file where the samples are stored
                        csv_file = "samples.csv",
                        # The field separator used in the CSV file
                        sep = ",",
                        # Whether to keep the original samples' names/
                        # indexes (if True, they are assumed to be in
                        # the first column of the data frame)
                        keep_samples_names = True,
                        # Whether to split the input data frame into
                        # two data frames, one containing only gene
                        # expression data and the other containing
                        # additional information about the samples
                        split = False)

INFO:bulkdgd.ioutil.samplesio:Since 'keep_samples_names = True', the samples will be identified using the values contained in the first column of the data frame.


INFO:bulkdgd.ioutil.samplesio:14895 column(s) containing gene expression data was (were) found in the input data frame.


## Split into training and test sets

Split the samples' IDs into a training set and a test set (50/50), using a fixed random seed for reproducibility.

In [5]:
# Split the samples' IDs into a training and a test set (50/50),
# using a fixed random seed for reproducibility.
rng = np.random.default_rng(seed = 42)
ids = df_samples_raw.index.to_numpy().copy()
rng.shuffle(ids)
split_point = len(ids) // 2
ids_train = ids[:split_point]
ids_test = ids[split_point:]

# Get the training and test samples as separate data frames.
df_train_raw = df_samples_raw.loc[ids_train]
df_test_raw = df_samples_raw.loc[ids_test]

## Preprocess the samples

Preprocess the training and test samples against a custom gene list, so that their genes match what the new model expects.

In [6]:
# Pre-process the training samples.
df_train, genes_excluded_train, genes_missing_train = \
    ioutil.preprocess_samples(df_samples = df_train_raw,
                              genes_txt_file = "custom_genes.txt")

# Pre-process the test samples.
df_test, genes_excluded_test, genes_missing_test = \
    ioutil.preprocess_samples(df_samples = df_test_raw,
                              genes_txt_file = "custom_genes.txt")

INFO:bulkdgd.ioutil.samplesio:14895 column(s) containing gene expression data was (were) found in the input data frame.


INFO:bulkdgd.ioutil.samplesio:Now looking for duplicated samples...


INFO:bulkdgd.ioutil.samplesio:No duplicated samples were found.


INFO:bulkdgd.ioutil.samplesio:Now looking for missing values in the columns containing gene expression data...


INFO:bulkdgd.ioutil.samplesio:No missing values were found in the columns containing gene expression data.


...

INFO:bulkdgd.ioutil.samplesio:No duplicated genes were found.


INFO:bulkdgd.ioutil.samplesio:In the data frame containing the pre-processed samples, the columns containing gene expression data will be ordered according to the list of genes included in the DGD model (taken from 'custom_genes.txt').


INFO:bulkdgd.ioutil.samplesio:In the data frame containing the pre-processed samples, the columns found in the input data frame which did not contain gene expression data, if any were present, will be appended as the last columns of the data frame and appear in the same order as they did in the input data frame.


INFO:bulkdgd.ioutil.samplesio:All genes found in the input samples are part of the set of genes included in the DGD model.


INFO:bulkdgd.ioutil.samplesio:All genes included in the DGD model were found in the input samples.


## Load the configuration for training

Load the configuration describing the training process (number of epochs, optimizers, learning rate schedules, early stopping, and which metrics/outputs to report).

In [7]:
# Load the configuration for training the model.
config_train = ioutil.load_config_train("training.yaml")

## Train the model

Train the model on the combined preprocessed samples, using the given training/test split, and unpack the resulting representations, predictions, losses, metrics, and timing information.

In [8]:
# Prepare combined samples and names for the train API.
# Concatenate preprocessed train and test samples into a single
# data frame.
df_samples_all = pd.concat([df_train, df_test])
names_train = df_train.index.tolist()
names_test = df_test.index.tolist()

# Train the bulkDGD model.
outputs = dgd_model.train(df_samples = df_samples_all,
                          names_train = names_train,
                          names_test = names_test,
                          config_train = config_train)

# Unpack the outputs.
((df_rep_train, df_rep_test),
    (df_pred_means_train, df_pred_means_test),
    pred_r_values,
    df_loss,
    dfs_metrics,
    df_time) = outputs

INFO:bulkdgd.core.model:Weight-threshold collapsed-component removal is enabled (threshold = 1e-08).


INFO:bulkdgd.core.model:Selected reporting metrics for the latent space: 'silhouette_score'.


INFO:bulkdgd.core.model:Epoch 1: loss train 147962.128, loss test 144613.444, epoch total CPU time 2.360 s, epoch total wall clock time 0.644 s, LR: decoder=4.10e-04, LR: rep_train=4.10e-04, noise scale 0.100000.


INFO:bulkdgd.core.model:Epoch 2: loss train 146193.756, loss test 142844.747, epoch total CPU time 0.029 s, epoch total wall clock time 0.030 s, LR: decoder=4.39e-04, LR: rep_train=4.39e-04, noise scale 0.099994.


INFO:bulkdgd.core.model:Epoch 3: loss train 144313.648, loss test 140676.903, epoch total CPU time 0.029 s, epoch total wall clock time 0.029 s, LR: decoder=4.89e-04, LR: rep_train=4.89e-04, noise scale 0.099975.


...

INFO:bulkdgd.core.model:Epoch 200: loss train 95660.923, loss test 103913.255, epoch total CPU time 0.032 s, epoch total wall clock time 0.032 s, LR: decoder=1.50e-06, LR: rep_train=1.50e-06, noise scale 0.001000.


INFO:bulkdgd.core.model:Epoch 200: silhouette_score: train=0.2949, test=0.2177.


INFO:bulkdgd.core.model:Restored best model state from epoch 200.


INFO:bulkdgd.core.model:The trained Gaussian mixture model's parameters were successfully saved in 'gmm.pth'.


INFO:bulkdgd.core.model:The trained decoder's parameters were successfully saved in 'dec.pth'.


## Save the outputs

Save the preprocessed samples, the representations found for the training and test samples, the decoder's predictions, the per-epoch losses and (if computed) clustering metrics, and the training time, all under an `output/` directory. The trained model's own parameters (`gmm.pth`/`dec.pth`) are saved separately by the `train()` call itself.

In [9]:
# Make a directory to save the outputs.
if not os.path.exists("output"):

    # Create the directory.
    os.mkdir("output")

# Save the pre-processed training samples.
ioutil.save_samples(\
   df = df_train,
   csv_file = "output/samples_preprocessed_train.csv",
   sep = ",")

# Save the pre-processed test samples.
ioutil.save_samples(\
   df = df_test,
   csv_file = "output/samples_preprocessed_test.csv",
   sep = ",")

# Save the representations for the training samples.
ioutil.save_representations(\
    df = df_rep_train,
    csv_file = "output/representations_train.csv",
    sep = ",")

# Save the representations for the test samples.
ioutil.save_representations(\
    df = df_rep_test,
    csv_file = "output/representations_test.csv",
    sep = ",")

# Save the predicted means for train and test.
ioutil.save_decoder_outputs(\
    df = df_pred_means_train,
    csv_file = "output/pred_means_train.csv",
    sep = ",")

ioutil.save_decoder_outputs(\
    df = df_pred_means_test,
    csv_file = "output/pred_means_test.csv",
    sep = ",")

# If there are two sets of predicted r-values (train and test)  
if isinstance(pred_r_values, tuple) and len(pred_r_values) == 2:

    # Unpack them.
    df_pred_r_values_train, df_pred_r_values_test = pred_r_values

    # Save them.
    ioutil.save_decoder_outputs(
        df = df_pred_r_values_train,
        csv_file = "output/pred_r_values_train.csv",
        sep = ",")

    ioutil.save_decoder_outputs(
        df = df_pred_r_values_test,
        csv_file = "output/pred_r_values_test.csv",
        sep = ",")

# Otherwise
else:

    # If it is a tuple of length 1, extract the single data frame
    df_r = \
        pred_r_values[0] if isinstance(pred_r_values, tuple) \
            else pred_r_values

    # Save the predicted r-values.
    ioutil.save_decoder_outputs(df = df_r,
                                csv_file = "output/pred_r_values.csv",
                                sep = ",")

# Save the losses.
df_loss.to_csv("output/loss.csv",
               sep = ",",
               index = False)

# If there are metrics.
if dfs_metrics is not None:

    # Unpack them.
    df_metrics_train, df_metrics_test = dfs_metrics

    # Save them.
    df_metrics_train.to_csv("output/metrics_train.csv",
                            sep = ",",
                            index = False)
    df_metrics_test.to_csv("output/metrics_test.csv",
                           sep = ",",
                           index = False)

# Save the time data.
df_time.to_csv("output/train_time.csv",
               sep = ",",
               index = False)